# 원시 가격 데이터 EDA

`data/raw/prices_raw.parquet`(yfinance 수집)을 직접 들여다본다. 셀을 위에서부터 실행(Shift+Enter)하며
표·차트로 값·분포·상관·이상치를 확인한다.

> 실행 전: `pip install -r requirements.txt -r requirements-dev.txt`, 그리고 `python -m src.data.collect`로 데이터 준비.

In [ ]:
import os, sys
# 저장소 루트 기준으로 경로 맞추기 (노트북이 notebooks/ 에 있을 때)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # HTML 내보내기 시 차트 포함

from src.config_loader import load_config, get_assets
from src.data.collect import load_raw
from src.data import validate as v

cfg = load_config()
assets = get_assets(cfg)
prices = load_raw(cfg["data"]["raw_dir"])
rets = np.log(prices / prices.shift(1)).iloc[1:]
print(prices.shape, assets)

## 1. 기본 정보 — 형태·기간·자료형

In [ ]:
print("기간:", prices.index.min().date(), "~", prices.index.max().date())
print("dtypes:", dict(prices.dtypes.astype(str)))
print("결측:", int(prices.isna().sum().sum()), "| 0이하 가격:", int((prices <= 0).sum().sum()))
prices.head()

In [ ]:
prices.describe().round(2)

## 2. 가격 추이 (로그 스케일)

자산별 스케일 차가 크므로 로그축으로 본다. SHV(현금성)는 거의 평평해야 정상.

In [ ]:
fig = px.line(prices, log_y=True, title="자산별 가격 추이 (로그 스케일)")
fig.update_layout(height=420)
fig

## 3. 일간 로그수익률 분포

꼬리 두께(첨도)·치우침(왜도) 확인. 주식은 음의 왜도·높은 첨도가 정상.

In [ ]:
long = rets.melt(var_name="asset", value_name="logret")
fig = px.histogram(long, x="logret", facet_col="asset", nbins=80,
                   title="일간 로그수익률 분포")
fig.update_layout(height=320, showlegend=False)
fig

In [ ]:
stats = pd.DataFrame({
    "연율수익률": rets.mean() * 252,
    "연율변동성": rets.std() * np.sqrt(252),
    "왜도": rets.skew(),
    "첨도": rets.kurtosis(),
}).round(3)
stats

## 4. 자산 간 상관 (수익률 기준)

SPY↔TLT 음(주식-채권 헤지), SHV는 무상관(현금성)이면 상식과 일치.

In [ ]:
corr = rets.corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu", zmin=-1, zmax=1,
                title="수익률 상관 히트맵")
fig.update_layout(height=420)
fig

## 5. 20일 롤링 변동성 (연율화)

In [ ]:
roll_vol = rets.rolling(20).std() * np.sqrt(252)
fig = px.line(roll_vol, title="20일 롤링 변동성 (연율화)")
fig.update_layout(height=380)
fig

## 6. 자동 요약 + 품질 검증

`validate.py`의 요약·검증을 호출. 이슈가 비어 있으면 통과.

In [ ]:
s = v.summarize_prices(prices)
print("극단 이동(|logret|>0.25):", s["extreme_moves"])
print("분할 미조정 의심:", s["split_suspect"])
print("영업일 대비 갭(공휴일 포함):", s["calendar_gaps"])

issues = v.validate_prices(prices, assets)
print("\n품질 검증:", "통과 ✅" if not issues else issues)